<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 7: </b>Retrieval-Augmented Generation with Vector Stores</h2>
<br>

In the previous notebook, we learned about embedding models and exercised some of their capabilities. We discussed their intended use cases of longer-form document comparison and found ways to use it as a backbone for more custom semantic comparisons. This notebook will progress these ideas toward the retrieval model's intended use case and explore how to build chatbot systems that rely on *vector stores* to automatically save and retrieve information.

<br>

### **Learning Objectives:**

- Understand how semantic-similarity-backed systems can facilitate easy-to-use retrieval formulations.

- Learn how to incorporate retrieval modules into your chat model systems for a retrieval-augmented generation (RAG) pipeline, which can be applied to tasks like document retrieval and conversation memory buffers.

<br>

### **Questions To Think About:**

- This notebook does not attempt to incorporate hierarchical reasoning or non-naive RAG (such as planning agents). Consider what modifications would be necessary to make these components work in an LCEL chain.

- Consider when it would be best to move your vector store solution into a scalable service and when a GPU will become necessary for optimization.

<br>

### **Environment Setup:**

In [2]:
# %%capture
## ^^ Comment out if you want to see the pip install process

## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints fastembed gradio rich
# %pip install -q arxiv pymupdf faiss-cpu
# !mkdir -p cached_papers && test -s cached_papers/2210.03629v3.pdf || wget -q --tries=3 --timeout=20 -O cached_papers/2210.03629v3.pdf https://arxiv.org/pdf/2210.03629v3

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [3]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

embedder = NVIDIAEmbeddings(
    model="course/embedding",
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

----

<br>

## Part 1: Summary of RAG Workflows

This notebook will explore several paradigms and derive reference code to help you approach some of the most common retrieval-augmented workflows. Specifically, the following sections will be covered (with the differences highlighted):

<br>

> ***Vector Store Workflow for Conversational Exchanges:***
- Generate semantic embedding for each new conversation.
- Add the message body to a vector store for retrieval.
- Query the vector store for relevant messages to fill in the LLM context.

<br>

> ***Modified Workflow for an Arbitrary Document:***
- **Divide the document into chunks and process them into useful messages.**
- Generate semantic embedding for each **new document chunk**.
- Add the **chunk bodies** to a vector store for retrieval.
- Query the vector store for relevant **chunks** to fill in the LLM context.
    - ***Optional:* Modify/synthesize results for better LLM results.**

<br>

> **Extended Workflow for a Directory of Arbitrary Documents:**
- Divide **each document** into chunks and process them into useful messages.
- Generate semantic embedding for each new document chunk.
- Add the chunk bodies to **a scalable vector database for fast retrieval**.
    - ***Optional*: Exploit hierarchical or metadata structures for larger systems.**
- Query the **vector database** for relevant chunks to fill in the LLM context.
    - *Optional:* Modify/synthesize results for better LLM results.

<br>

Some of the most important terminology surrounding RAG is covered in detail on the [**LlamaIndex Concepts page**](https://developers.llamaindex.ai/python/framework/getting_started/concepts/), which itself is a great starting point for progressing towards the LlamaIndex loading and retrieving strategy. We highly recommend using it as a reference as you continue with this notebook and advise you to try out LlamaIndex after the course to consider the pros and cons firsthand!


<!-- > <img src="https://drive.google.com/uc?export=view&id=1cFbKbVvLLnFPs3yWCKIuzXkhBWh6nLQY" width=1200px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/data_connection_langchain.jpeg" width=1200px/>
>
> From [**Retrieval | LangChain**🦜️🔗](https://blog.langchain.com/syncing-data-sources-to-vector-stores/)

----

<br>

## **Part 2:** RAG for Conversation History

In our previous explorations, we delved into the capabilities of document embedding models and used them to embed, store, and compare semantic vector representations of text. Though we could motivate how to efficiently extend this into vector store land manually, the true beauty of working with a standard API is its strong incorporation with other frameworks that can already do the heavy lifting for us!

<br>

### **Step 1**: Getting A Conversation

Consider a conversation crafted using Llama-13B between a chat agent and a blue bear named Beras. This dialogue, dense with details and potential diversions, provides a rich dataset for our study:


In [4]:
conversation = [  ## This conversation was generated partially by an AI system, and modified to exhibit desirable properties
    "[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?",
    "[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America",
    "[Beras] Wow, that sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard many great things about them.",
    "[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you!",
    "[Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.",
    "[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online or watching documentaries about them.",
    "[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just curious, ya know!",
    "[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains and their significance!"
]

Using the manual embedding strategy from the previous notebook is still very viable, but we can also rest easy and let a **vector store** do all that work for us!

<br>

### **Step 2:** Constructing Our Vector Store Retriever

To streamline similarity queries on our conversation, we can employ a vector store to help keep track of passages for us! **Vector Stores**, or vector storage systems, abstract away most of the low-level details of the embedding/comparison strategies and provide a simple interface to load and compare vectors.


<!-- > <img src="https://drive.google.com/uc?export=view&id=1ZjwYbSZzsXK6ZP8O1-cY3BeRffV4oqzb" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/vector_stores.jpeg" width=1200px/>
>
> From [**Vector Stores | LangChain**🦜️🔗](https://blog.langchain.com/syncing-data-sources-to-vector-stores/vectorstores/)

<br>

In addition to simplifying the process from an API perspective, vector stores also implement connectors, integrations, and optimizations under the hood. In our case, we will start with the [**FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss), which integrates a LangChain-compatable Embedding model with the [**FAISS (Facebook AI Similarity Search)**](https://github.com/facebookresearch/faiss) library to make the process fast and scalable on our local machine!

**Specifically:**

1. We can feed our conversation into [**a FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss) via the `from_texts` constructor. This will take our conversational data and the embedding model to create a searchable index over our discussion.
2. This vector store can then be "interpreted" as a retriever, supporting the LangChain runnable API and returning documents retrieved via an input query.

The following shows how you can construct a FAISS vector store and reinterpret it as a retriever using the LangChain `vectorstore` API:

In [5]:
%%time
## ^^ This cell will be timed to see how long the conversation embedding takes
from langchain_community.vectorstores import FAISS

## Streamlined from_texts FAISS vectorstore construction from text list
convstore = FAISS.from_texts(conversation, embedding=embedder)
retriever = convstore.as_retriever()

CPU times: user 479 ms, sys: 24.9 ms, total: 504 ms
Wall time: 711 ms


The retriever can now be used like any other LangChain runnable to query the vector store for some relevant documents:

In [6]:
pprint(retriever.invoke("What is your name?"))

[
    Document(
        id='aad4f41b-c073-458f-a3ce-32b967471b9a',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='e3d233c5-d2f4-4095-8309-42d1c931ebae',
        metadata={},
        page_content='[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you!'
    ),
    Document(
        id='404e5081-00dc-4b6b-b565-84067004d453',
        metadata={},
        page_content="[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just curious, 
ya know!"
    ),
    Document(
        id='dd9cc5e7-aa93-45d0-86c2-800364515898',
        metadata={},
        page_content='[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains
and their significance!'
    )
]

In [7]:
pprint(retriever.invoke("Where are the Rocky Mountains?"))

[
    Document(
        id='aad4f41b-c073-458f-a3ce-32b967471b9a',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='b0e13f29-38bb-404d-b640-7952844d3b9b',
        metadata={},
        page_content='[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch 
across North America'
    ),
    Document(
        id='4b55a7fa-b814-4769-9eb9-667a586eb8bc',
        metadata={},
        page_content='[Beras] Wow, that sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard
many great things about them.'
    ),
    Document(
        id='caf82e9c-ec32-437b-89c7-4825caa78a46',
        metadata={},
        page_content='[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research 
online or watching documentaries about them.'
    )
]

As we can see, our retriever found a handful of semantically relevant documents from our query. You may notice that not all of the documents are useful or clear on their own. For example, a retrieval of *"Beras"* for *"your name"* may be problematic for the chatbot if provided out of context. Anticipating the potential problems and creating synergies between your LLM components can increase the likelihood of good RAG behavior, so keep an eye out for such pitfalls and opportunities.

<br>

### **Optional Step:** Reranking Retrieved Candidates

The vector store compares compact representations, which makes it practical to search many stored documents. Once it has produced a small candidate set, a cross-encoder can examine the query together with each candidate and refine their order. This adds work at query time, so it is most useful after the vector search has narrowed the collection.

For this second step, the course model server keeps `cross-encoder/ms-marco-MiniLM-L6-v2` loaded on CPU and exposes it through the NVIDIA ranking interface. `NVIDIARerank` therefore uses the same client boundary as a hosted or separately deployed NVIDIA NIM. In Colab, the connector can target a compatible NVIDIA reranking endpoint by changing the model and base URL after you provide an API key.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIARerank

rerank_query = "Where are the Rocky Mountains?"
candidates = retriever.invoke(rerank_query)
reranker = NVIDIARerank(
    model="course/reranker",
    base_url="http://llm_client:9000/v1",
    top_n=len(candidates),
)
reranked = reranker.compress_documents(candidates, query=rerank_query)

for rank, document in enumerate(reranked, start=1):
    score = document.metadata["relevance_score"]
    print(f"{rank}. {score:.3f}  {document.page_content}")

### **Step 3:** Incorporating Conversation Retrieval Into Our Chain

Now that we have our loaded retriever component as a chain, we can incorporate it into our existing chat system as before. Specifically, we can start with an ***always-on RAG formulation*** where:
- **A retriever is always retrieving context by default**.
- **A generator is acting on the retrieved context**.

In [8]:
from langchain_community.document_transformers import LongContextReorder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from functools import partial
from operator import itemgetter

########################################################################
## Utility Runnables/Methods
def RPrint(preface=""):
    """Simple passthrough "prints, then returns" chain"""
    def print_and_return(x, preface):
        if preface: print(preface, end="")
        pprint(x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name:
            out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

## Optional; Reorders longer documents to center of output text
long_reorder = RunnableLambda(LongContextReorder().transform_documents)

In [9]:
context_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {question}"
    "\nAnswer the user conversationally. User is not aware of context."
)

chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'question': (lambda x:x)
    }
    | context_prompt
    # | RPrint()
    | instruct_llm
    | StrOutputParser()
)

pprint(chain.invoke("Where does Beras live?"))

Beras lives in the arctic!

Take a second to try out some more invocations and see how the new setup performs. Regardless of your model choice, the following questions should serve as interesting starting points.

In [10]:
pprint(chain.invoke("Where are the Rocky Mountains?"))

The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America! You can 
learn more about them by researching online or watching documentaries if you're curious about the details.

In [11]:
pprint(chain.invoke("Where are the Rocky Mountains? Are they close to California?"))

Exception: [504] {'message': 'Provider request failed', 'type': 'upstream_error'}
{'error': {'message': 'Provider request failed', 'type': 'upstream_error'}, 'dli_routing': {'reason': 'ttft_timeout', 'model': 'nvidia/nemotron-3.5-lightning-30b-a3b', 'effective_route': 'primary', 'primary': {'state': 'closed', 'consecutive_failures': 1, 'tier': 0, 'retry_after_seconds': 0, 'probe_in_flight': False, 'last_failure': 'ttft_timeout', 'attempts': 37, 'successes': 35, 'failed_requests': 2, 'available': True, 'model_allowed': True}, 'proxy_instance_id': '006286a03c304b03927bf7e54aaeaa4f', 'course_cache_sha256': 'a266e086ee562154bd6c74bb6c609c3420f84ee51a80c7d2a0ae4815abf957f1', 'fallback': None, 'policy': {'failure_threshold': 3, 'base_cooldown_seconds': 30.0, 'max_cooldown_seconds': 300.0, 'scope': 'provider_and_model', 'persistence': 'proxy_process', 'recovery_probes': 1}}, 'status': 504}

In [ ]:
pprint(chain.invoke("How far away is Beras from the Rocky Mountains?"))

<br>

You might notice some decent performance with this always-on retrieval node in the loop since the actual context being fed into the LLM remains relatively small. It's important to experiment with factors like embedding sizes, context limits, and model options to see what kinds of behavior you can expect and which efforts are worth taking to improve performance.

<br>

### **Step 4:** Automatic Conversation Storage

Now that we see how our vector store memory unit should function, we can perform one last integration to allow our conversation to add new entries to our conversation: a runnable that calls the `add_texts` method for us to update the store state.


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

########################################################################
## Reset knowledge base and define what it means to add more messages.
convstore = FAISS.from_texts(conversation, embedding=embedder)

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([f"User said {d.get('input')}", f"Agent said {d.get('output')}"])
    return d.get('output')

########################################################################

# instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

chat_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {input}"
    "\nAnswer the user conversationally. Make sure the conversation flows naturally.\n"
    "[Agent]"
)


conv_chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'input': (lambda x:x)
    }
    | RunnableAssign({'output' : chat_prompt | instruct_llm | StrOutputParser()})
    | partial(save_memory_and_get_output, vstore=convstore)
)

pprint(conv_chain.invoke("I'm glad you agree! I can't wait to get some ice cream there! It's such a good food!"))
print()
pprint(conv_chain.invoke("Can you guess what my favorite food is?"))
print()
pprint(conv_chain.invoke("Actually, my favorite is honey! Not sure where you got that idea?"))
print()
pprint(conv_chain.invoke("I see! Fair enough! Do you know my favorite food now?"))

Exception: [504] {'message': 'Provider request failed', 'type': 'upstream_error'}
{'error': {'message': 'Provider request failed', 'type': 'upstream_error'}, 'dli_routing': {'reason': 'ttft_timeout', 'model': 'nvidia/nemotron-3.5-lightning-30b-a3b', 'effective_route': 'primary', 'primary': {'state': 'closed', 'consecutive_failures': 2, 'tier': 0, 'retry_after_seconds': 0, 'probe_in_flight': False, 'last_failure': 'ttft_timeout', 'attempts': 38, 'successes': 35, 'failed_requests': 3, 'available': True, 'model_allowed': True}, 'proxy_instance_id': '006286a03c304b03927bf7e54aaeaa4f', 'course_cache_sha256': 'a266e086ee562154bd6c74bb6c609c3420f84ee51a80c7d2a0ae4815abf957f1', 'fallback': None, 'policy': {'failure_threshold': 3, 'base_cooldown_seconds': 30.0, 'max_cooldown_seconds': 300.0, 'scope': 'provider_and_model', 'persistence': 'proxy_process', 'recovery_probes': 1}}, 'status': 504}

Unlike the more automatic full-text or rule-based approaches to injecting context into the LLM, this approach ensures some amount of consolidation which can keep the context length from getting out of hand. It's still not a full-proof strategy on its own, but it's a stark improvement for unstructured conversations (and doesn't even require a strong instruction-tuned model to perform slot-filling).

----

<br>

## **Part 3 [Exercise]:** RAG For Document Chunk Retrieval

Given our prior exploration of document loading, the idea that data chunks can be embedded and searched through probably isn't surprising. With that said, it is definitely worth going over since applying RAG with documents is a double-edged sword; it may **seem** to work well out of the box but requires some extra care when optimizing it for truly reliable performance. It also provides an excellent opportunity to review some fundamental LCEL skills, so let's see what we can do!

<br>

### **Exercise:**

In the previous example, you may recall that we pulled in some relatively small papers with the help of [`ArxivLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/arxiv/ArxivLoader) using the following syntax:

```python
from langchain_community.document_loaders import ArxivLoader

docs = [
    ArxivLoader(query="2205.00445").load(),  ## MRKL
    ArxivLoader(query="2210.03629").load(),  ## ReAct
]
```

Given all that you've learned so far, choose a selection of papers that you would like to use and develop a chatbot that can talk about them!

<br>

Though this is a pretty big task, a walkthrough of ***most*** of the process will be provided below. By the end of the walkthrough, many of the necessary puzzle pieces will be provided, and your real task will be to integrate them together for the final `retrieval_chain`. When you're done, get ready to re-integrate the chain (or a flavor of your choice) in the last notebook as part of the evaluation exercise!


<br>

### **Task 1**: Loading And Chunking Your Documents

The following code block gives you some default papers to load in for your RAG chain. Feel free to select more papers as desired, but note that longer documents will take longer to process. A few simplifying assumptions and additional processing steps are included to help you improve your naive RAG performance:

- Documents are cut off prior to the "References" section if one exists. This will keep our system from considering the citations and appendix sections, which tend to be long and distracting.

- A chunk that lists the available documents is inserted to provide a high-level view of all available documents in a single chunk. If your pipeline does not provide metadata on each retrieval, this is a useful component and can even be listed among a list of higher-priority pieces if appropriate.

- Additionally, the metadata entries are also inserted to provide general information. Ideally, there would also be some synthetic chunks that merge the metadata into interesting cross-document chunks.



In [13]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import ArxivLoader, PyMuPDFLoader
from langchain_community.retrievers import ArxivRetriever
from langchain_core.documents import Document
from IPython.display import display, Markdown
from pathlib import Path
from composer.documents import load_pdf, validate_documents

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)    

## TODO: Please pick some papers and add them to the list as you'd like
print("Loading Documents")
paper_ids = [
    # "1706.03762",  ## Attention Is All You Need — transformer foundation
    # "1810.04805",  ## BERT — retrieval/embedding foundation
    # "2103.00020",  ## CLIP — multimodal representation learning
    # "2112.10752",  ## Latent Diffusion — generative modeling milestone
    "2005.11401",  ## Original RAG paper
    "2210.03629",  ## ReAct — reasoning + acting / agent foundation
    # "2310.06825",  ## Mistral 7B — strong modern open LLM
    "2306.05685",  ## Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena

    "2602.06176",  ## Large Language Model Reasoning Failures — Feb 2026 survey
    # "2601.01828",  ## Emergent Introspective Awareness in Large Language Models
    "2603.18272",  ## Retrieval-Augmented LLM Agents: Learning to Learn from 
]
try:
    docs = [doc[0] for doc in ArxivRetriever(get_full_documents=True, doc_content_chars_max=1000000).batch(paper_ids, config={"max_concurrency": 1})]
    validate_documents(docs)
except Exception as error:
    cached_paper = Path("cached_papers/2210.03629v3.pdf")
    if not cached_paper.is_file():
        raise RuntimeError("Live arXiv retrieval failed and the cached ReAct paper is missing.") from error
    print("Live arXiv retrieval was unavailable; loading a cached ReAct paper to keep the workflow available.")
    cached_pages = load_pdf(str(cached_paper))
    docs = [Document(
        page_content="\n\n".join(page.page_content for page in cached_pages),
        metadata={"Title": "ReAct: Synergizing Reasoning and Acting in Language Models", "entry_id": "2210.03629v3", "Summary": cached_pages[0].metadata.get("Summary", "")},
    )]

validate_documents(docs)

## Cut the paper short if references is included.
## This is a standard string in papers.
for doc in docs:
    content = doc.page_content
    if "References" in content:
        doc.page_content = content[:content.index("References")]

## Split the documents and also filter out stubs (overly short chunks)
print("Chunking Documents")
docs_chunks = [text_splitter.split_documents([doc]) for doc in docs]
docs_chunks = [[c for c in dchunks if len(c.page_content) > 200] for dchunks in docs_chunks]

## Make some custom Chunks to give big-picture details
doc_string = "Available Documents:"
doc_metadata = []
for chunks in docs_chunks:
    metadata = getattr(chunks[0], 'metadata', {})
    doc_string += "\n - " + metadata.get('Title')
    doc_metadata += [str(metadata)]

extra_chunks = [doc_string] + doc_metadata

## Printing out some summary information for reference
pprint(doc_string, '\n')
for i, chunks in enumerate(docs_chunks):
    print(f"Document {i}")
    print(f" - # Chunks: {len(chunks)}")
    print(f" - Metadata: ")
    pprint(chunks[0].metadata)
    preview = chunks[0].page_content[:2000]
    display(Markdown(f"<details><summary>Chunk 0</summary>\n\n{preview}\n\n</details>"))
    print()

Loading Documents
Live arXiv retrieval was unavailable; loading a cached ReAct paper to keep the workflow available.
Chunking Documents


Available Documents:
 - ReAct: Synergizing Reasoning and Acting in Language Models 

Document 0
 - # Chunks: 147
 - Metadata: 


{
    'Title': 'ReAct: Synergizing Reasoning and Acting in Language Models',
    'entry_id': '2210.03629v3',
    'Summary': 'While large language models (LLMs) have demonstrated impressive performance across tasks in 
language understanding and interactive decision making, their abilities for reasoning (e.g. chain-of-thought 
prompting) and acting (e.g. action plan generation) have primarily been studied as separate topics. In this paper, 
we explore the use of LLMs to generate both reasoning traces and task-speciﬁc actions in an interleaved manner, 
allowing for greater synergy between the two: reasoning traces help the model induce, track, and update action 
plans as well as handle exceptions, while actions allow it to interface with and gather additional information from
external sources such as knowledge bases or environments. We apply our approach, named ReAct, to a diverse set of 
language and decision making tasks and demonstrate its effectiveness over state-of-the-art baselines in addition to
improved human interpretability and trustworthiness. Concretely, on question answering (HotpotQA) and fact 
veriﬁcation (Fever), ReAct overcomes prevalent issues of hallucination and error propagation in chain-of-thought 
reasoning by interacting with a simple Wikipedia API, and generating human-like task-solving trajectories that are 
more interpretable than baselines without reasoning traces. Furthermore, on two interactive decision making 
benchmarks (ALFWorld and WebShop), ReAct outperforms imitation and reinforcement learning methods by an absolute 
success rate of 34% and 10% respectively, while being prompted with only one or two in-context examples.'
}

<details><summary>Chunk 0</summary>

Published as a conference paper at ICLR 2023
REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
Shunyu Yao∗*,1, Jeffrey Zhao2, Dian Yu2, Nan Du2, Izhak Shafran2, Karthik Narasimhan1, Yuan Cao2
1Department of Computer Science, Princeton University
2Google Research, Brain team
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
While large language models (LLMs) have demonstrated impressive performance
across tasks in language understanding and interactive decision making, their
abilities for reasoning (e.g. chain-of-thought prompting) and acting (e.g. action
plan generation) have primarily been studied as separate topics. In this paper, we
explore the use of LLMs to generate both reasoning traces and task-speciﬁc actions
in an interleaved manner, allowing for greater synergy between the two: reasoning
traces help the model induce, track, and update action plans as well as handle

</details>

<br>

### **Task 2**: Construct Your Document Vector Stores

Now that we have all of the components, we can go ahead and create indices surrounding them:

In [14]:
%%time
print("Constructing Vector Stores")
vecstores = [FAISS.from_texts(extra_chunks, embedder)]
vecstores += [FAISS.from_documents(doc_chunks, embedder) for doc_chunks in docs_chunks]

Constructing Vector Stores
CPU times: user 41.5 ms, sys: 19.3 ms, total: 60.8 ms
Wall time: 9.01 s


<br>

From there, we can combine our indices into a single one using the following utility:

In [15]:
from faiss import IndexFlatL2
from langchain_community.docstore.in_memory import InMemoryDocstore

embed_dims = len(embedder.embed_query("test"))
def default_FAISS():
    '''Useful utility for making an empty FAISS vectorstore'''
    return FAISS(
        embedding_function=embedder,
        index=IndexFlatL2(embed_dims),
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
        normalize_L2=False
    )

def aggregate_vstores(vectorstores):
    ## Initialize an empty FAISS Index and merge others into it
    ## We'll use default_faiss for simplicity, though it's tied to your embedder by reference
    agg_vstore = default_FAISS()
    for vstore in vectorstores:
        agg_vstore.merge_from(vstore)
    return agg_vstore

## Unintuitive optimization; merge_from seems to optimize constituent vector stores away
docstore = aggregate_vstores(vecstores)

print(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")

Constructed aggregate docstore with 149 chunks


<br>

### **Task 3: [Exercise]** Implement Your RAG Chain

Finally, all the puzzle pieces are in place to implement the RAG pipeline! As a review, we now have:

- A way to construct a from-scratch vector store for conversational memory (and a way to initialize an empty one with `default_FAISS()`)

- A vector store pre-loaded with useful document information from our `ArxivLoader` utility (stored in `docstore`).

With the help of a couple more utilities, you're finally ready to integrate your chain! A few additional convenience utilities are provided (`doc2str` and the now-common `RPrint`) but are optional to use. Additionally, some starter prompts and structures are also defined.

> **Given all of this:** Please implement the `retrieval_chain`.

In [16]:
from langchain_community.document_transformers import LongContextReorder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import gradio as gr
from functools import partial
from operator import itemgetter

instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

convstore = default_FAISS()

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([
        f"User previously responded with {d.get('input')}",
        f"Agent previously responded with {d.get('output')}"
    ])
    return d.get('output')

initial_msg = (
    "Hello! I am a document chat agent here to help the user!"
    f" I have access to the following documents: {doc_string}\n\nHow can I help you?"
)

chat_prompt = ChatPromptTemplate.from_messages([("system",
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked: {input}\n\n"
    " From this, we have retrieved the following potentially-useful info: "
    " Conversation History Retrieval:\n{history}\n\n"
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational.)"
), ('user', '{input}')])

stream_chain = chat_prompt | instruct_llm | StrOutputParser()

################################################################################################
## BEGIN TODO: Implement the retrieval chain to make your system work!

retrieval_chain = (
    {'input': (lambda x: x)}
    | RunnableAssign({
        'history': (
            itemgetter('input')
            | convstore.as_retriever()
            | long_reorder
            | docs2str
        )
    })
    | RunnableAssign({
        'context': (
            itemgetter('input')
            | docstore.as_retriever()
            | long_reorder
            | docs2str
        )
    })
)

## END TODO
################################################################################################

def chat_gen(message, history=None, return_buffer=True):
    history = [] if history is None else history
    buffer = ""
    ## First perform the retrieval based on the input message
    retrieval = retrieval_chain.invoke(message)
    line_buffer = ""

    ## Then, stream the results of the stream_chain
    for token in stream_chain.stream(retrieval):
        buffer += token
        ## If you're using standard print, keep line from getting too long
        yield buffer if return_buffer else token

    ## Lastly, save the chat exchange to the conversation memory buffer
    save_memory_and_get_output({'input':  message, 'output': buffer}, convstore)


## Start of Agent Event Loop
test_question = "Tell me about RAG!"  ## <- modify as desired

## Before you launch your gradio interface, make sure your thing works
for response in chat_gen(test_question, return_buffer=False):
    print(response, end='')

Exception: [504] {'message': 'Provider request failed', 'type': 'upstream_error'}
{'error': {'message': 'Provider request failed', 'type': 'upstream_error'}, 'dli_routing': {'reason': 'ttft_timeout', 'model': 'nvidia/nemotron-3.5-lightning-30b-a3b', 'effective_route': 'unavailable', 'primary': {'state': 'open', 'consecutive_failures': 3, 'tier': 1, 'retry_after_seconds': 30.0, 'probe_in_flight': False, 'last_failure': 'ttft_timeout', 'attempts': 39, 'successes': 35, 'failed_requests': 4, 'available': False, 'model_allowed': True}, 'proxy_instance_id': '006286a03c304b03927bf7e54aaeaa4f', 'course_cache_sha256': 'a266e086ee562154bd6c74bb6c609c3420f84ee51a80c7d2a0ae4815abf957f1', 'fallback': None, 'policy': {'failure_threshold': 3, 'base_cooldown_seconds': 30.0, 'max_cooldown_seconds': 300.0, 'scope': 'provider_and_model', 'persistence': 'proxy_process', 'recovery_probes': 1}}, 'status': 504}

### **Task 4:** Interact With Your Gradio Chatbot

In [ ]:
# chatbot = gr.Chatbot(value = [{"role": "user", "content": initial_msg}])
# demo = gr.ChatInterface(chat_gen, chatbot=chatbot).queue()

# try:
#     demo.launch(debug=True, share=False, show_api=False)
#     demo.close()
# except Exception as e:
#     demo.close()
#     print(e)
#     raise e

<br>

----

<br>

## **Part 4:** Saving Your Index For Evaluation

After you've implemented your RAG chain, please save your accumulated vector store as shown [in the official documentation](https://python.langchain.com/docs/integrations/vectorstores/faiss#saving-and-loading). You'll have a chance to use it again for your final assessment!

In [17]:
## Save and compress your index
validate_documents(list(docstore.docstore._dict.values()))
docstore.save_local("docstore_index")
!tar czvf docstore_index.tgz docstore_index

!rm -rf docstore_index

docstore_index/
docstore_index/index.pkl
docstore_index/index.faiss


If everything was properly saved, the following line can be invoked to pull the index from the compressed `tgz` file (assuming the pip requirements are installed). After you have confirmed that the cell can pull in your index, download `docstore_index.tgz` for use in the last notebook!

In [18]:
from langchain_community.vectorstores import FAISS

!tar xzvf docstore_index.tgz
new_db = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
validate_documents(list(new_db.docstore._dict.values()))
docs = new_db.similarity_search("Testing the index")
print(docs[0].page_content[:1000])

docstore_index/
docstore_index/index.pkl
docstore_index/index.faiss
(1d) ReAct (Reason + Act)
Thought 1: I need to search Cirque du Soleil show Mystere, find the hotel it is in, then find the number of rooms in the hotel.
Act 1: Search[Cirque du Soleil show Mystere ]
Obs 1: Could not find Cirque du Soleil show Mystere. Similar: ['Cirque du Soleil' ...
Thought 2: [green highlight: To find the hotel, I can search Mystere (Cirque du Soleil)].
Act 2: Search[Mystere (Cirque du Soleil) ]
Obs 2: Mystère (my-steer) is ... Held in a custom theatre at the Treasure Island Hotel and Casino ...
Thought 3: Mystere is held in a custom theatre at the Treasure Island Hotel and Casino. So I need to search Treasure Island Hotel and Casino next and find the number of rooms in the hotel.
Act 3: Search[[green highlight: Treasure Island Hotel and Casino]]
Obs 3: Treasure Island Hotel and Casino ... with 2,884 rooms and 220 suites ...
Thought 4: [green highlight: Treasure Island Hotel and Casino has 2,884 roo

-----

<br>

## **Part 5:** Wrap-Up

Congratulations! Assuming your RAG chain is all good, you're now ready to move on to the **RAG Evaluation [Assessment]** section!

### <font color="#76b900">**Great Job!**</font>

### **Next Steps:**
1. **[Optional]** Revisit the **"Questions To Think About" Section** at the top of the notebook and think about some possible answers.

---

<div style="width: 55%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>